In [6]:
from google.colab import files

uploaded = files.upload()

Saving project.txt to project.txt


# GroupDNA – WhatsApp Group Analytics

Name:Srushti   
Batch:Sept batch


In [5]:
import numpy as np
from datetime import datetime, timedelta

print("GroupDNA started successfully!")

GroupDNA started successfully!


# 1. WhatsApp Chat Parser

This section reads the WhatsApp TXT file and extracts the timestamp, sender and message. It also handles system messages, media messages, deleted messages and multiline messages.

# 2. Group Overview

In [8]:
FILE_NAME = "project.txt"

with open(FILE_NAME, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total lines:", len(lines))
print("\nFirst 5 lines:\n")

for line in lines[:5]:
    print(line.strip())

Total lines: 57

First 5 lines:

04/05/24, 09:15 - Aarav: Good morning everyone
04/05/24, 09:17 - Meera: Good morning!
04/05/24, 09:20 - Rohan: What is the plan for today?
04/05/24, 09:22 - Ananya: We have the project meeting at 11
04/05/24, 09:25 - Kabir: Okay, I will join


In [9]:
from datetime import datetime

def get_timestamp(line):

    if len(line) < 18:
        return None

    try:
        return datetime.strptime(line[:15], "%d/%m/%y, %H:%M")
    except ValueError:
        return None


messages = []
current_message = None

for raw_line in lines:

    line = raw_line.rstrip("\n")

    timestamp = get_timestamp(line)

    if timestamp is not None:

        if current_message is not None:
            messages.append(current_message)

        content = line[18:]

        if ": " in content:

            sender, text = content.split(": ", 1)

            current_message = {
                "timestamp": timestamp,
                "sender": sender,
                "text": text
            }

    else:

        if current_message is not None:
            current_message["text"] += " " + line.strip()


if current_message is not None:
    messages.append(current_message)


print("Total messages:", len(messages))

Total messages: 57


Display participants

In [10]:
participants = sorted(
    set(message["sender"] for message in messages)
)

print("Participants:")
for person in participants:
    print("-", person)

Participants:
- Aarav
- Ananya
- Kabir
- Meera
- Rohan


Calculate messages per person

In [11]:
participant_counts = {}

for person in participants:
    participant_counts[person] = 0

for message in messages:
    participant_counts[message["sender"]] += 1


print("MESSAGES PER PARTICIPANT")
print("------------------------")

for person, count in participant_counts.items():
    print(person, ":", count)

MESSAGES PER PARTICIPANT
------------------------
Aarav : 12
Ananya : 12
Kabir : 11
Meera : 12
Rohan : 10


Find the busiest day

In [12]:
day_counts = {}

for message in messages:

    date = message["timestamp"].date()

    if date not in day_counts:
        day_counts[date] = 0

    day_counts[date] += 1


busiest_day = max(day_counts, key=day_counts.get)

print("Busiest Day:", busiest_day)
print("Messages:", day_counts[busiest_day])

Busiest Day: 2024-05-04
Messages: 34


Find the busiest hour

In [13]:
hour_counts = {}

for message in messages:

    hour = message["timestamp"].hour

    if hour not in hour_counts:
        hour_counts[hour] = 0

    hour_counts[hour] += 1


busiest_hour = max(hour_counts, key=hour_counts.get)

print(
    "Busiest Hour:",
    f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00"
)

print("Messages:", hour_counts[busiest_hour])

Busiest Hour: 14:00 - 15:00
Messages: 13


In [14]:
import numpy as np

participant_index = {}

for index, person in enumerate(participants):
    participant_index[person] = index


activity_matrix = np.zeros(
    (len(participants), 24),
    dtype=int
)


for message in messages:

    person = message["sender"]
    hour = message["timestamp"].hour

    row = participant_index[person]

    activity_matrix[row][hour] += 1


print("Activity Matrix")
print("----------------")

print("Shape:", activity_matrix.shape)
print(activity_matrix)

Activity Matrix
----------------
Shape: (5, 24)
[[0 0 0 0 0 0 0 0 1 2 2 0 1 1 3 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 2 1 2 1 0 3 1 0 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 2 1 1 1 1 3 0 0 0 0 0 0 0 2 0]
 [0 0 0 0 0 0 0 0 1 1 2 1 1 1 2 2 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 2 1 2 1 0 2 1 0 0 1 0 0 0 0 0]]


Create a simple text heatmap

In [15]:
print("\nACTIVITY HEATMAP")
print("----------------")

print("        ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()


for row_index, person in enumerate(participants):

    print(f"{person:8}", end="")

    row = activity_matrix[row_index]

    maximum = np.max(row)

    for value in row:

        if value == 0:
            symbol = "."

        elif value <= maximum * 0.5:
            symbol = "░"

        else:
            symbol = "█"

        print(f"{symbol}  ", end="")

    print()


ACTIVITY HEATMAP
----------------
        00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aarav   .  .  .  .  .  .  .  .  ░  █  █  .  ░  ░  █  .  .  .  █  .  .  .  .  .  
Ananya  .  .  .  .  .  .  .  .  .  █  ░  █  ░  .  █  ░  .  .  ░  .  .  .  ░  .  
Kabir   .  .  .  .  .  .  .  .  .  █  ░  ░  ░  ░  █  .  .  .  .  .  .  .  █  .  
Meera   .  .  .  .  .  .  .  .  ░  ░  █  ░  ░  ░  █  █  .  .  ░  .  .  .  .  .  
Rohan   .  .  .  .  .  .  .  .  .  █  ░  █  ░  .  █  ░  .  .  ░  .  .  .  .  .  


Find top words

In [16]:
stop_words = {
    "the", "is", "a", "an", "to", "of", "in",
    "on", "for", "and", "or", "i", "you",
    "we", "it", "this", "that", "are",
    "am", "was", "be", "my", "your",
    "will", "have", "has", "do", "did"
}


word_counts = {}


for message in messages:

    words = message["text"].lower().split()

    for word in words:

        clean_word = ""

        for character in word:

            if character.isalnum():
                clean_word += character

        if len(clean_word) > 1 and clean_word not in stop_words:

            if clean_word not in word_counts:
                word_counts[clean_word] = 0

            word_counts[clean_word] += 1


top_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)


print("TOP 10 WORDS")
print("------------")

for word, count in top_words[:10]:
    print(word, ":", count)

TOP 10 WORDS
------------
good : 7
everyone : 5
morning : 4
what : 3
today : 3
meeting : 3
yes : 3
lets : 3
night : 3
project : 2


In [17]:
response_times = {}

for person in participants:
    response_times[person] = []


for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    if previous["sender"] != current["sender"]:

        difference = (
            current["timestamp"] -
            previous["timestamp"]
        ).total_seconds() / 60

        if 0 <= difference <= 1440:

            response_times[
                current["sender"]
            ].append(difference)


print("AVERAGE RESPONSE TIME")
print("---------------------")


for person in participants:

    times = response_times[person]

    if len(times) > 0:

        average = sum(times) / len(times)

    else:

        average = 0

    print(
        person,
        ":",
        round(average, 2),
        "minutes"
    )

AVERAGE RESPONSE TIME
---------------------
Aarav : 102.73 minutes
Ananya : 4.83 minutes
Kabir : 54.73 minutes
Meera : 12.18 minutes
Rohan : 29.1 minutes


In [18]:
all_dates = sorted(
    set(
        message["timestamp"].date()
        for message in messages
    )
)


for person in participants:

    active_dates = set()

    for message in messages:

        if message["sender"] == person:

            active_dates.add(
                message["timestamp"].date()
            )


    silent_days = 0

    for date in all_dates:

        if date not in active_dates:
            silent_days += 1


    print(
        person,
        ":",
        silent_days,
        "silent days"
    )

Aarav : 0 silent days
Ananya : 0 silent days
Kabir : 0 silent days
Meera : 0 silent days
Rohan : 0 silent days


In [19]:
print("=" * 60)
print("             GROUPDNA")
print("        CAMPUS GROUP ANALYTICS")
print("=" * 60)

print("\nGROUP OVERVIEW")
print("-" * 60)

print("Total Messages :", len(messages))
print("Participants   :", len(participants))
print("First Date     :", min(
    m["timestamp"] for m in messages
).date())

print("Last Date      :", max(
    m["timestamp"] for m in messages
).date())


print("\nMESSAGE COUNT")
print("-" * 60)

for person, count in participant_counts.items():
    print(f"{person:10} : {count}")


print("\nACTIVITY PEAK")
print("-" * 60)

print(
    "Busiest Day  :",
    busiest_day,
    "(",
    day_counts[busiest_day],
    "messages)"
)

print(
    "Busiest Hour :",
    f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00"
)


print("\nTOP WORDS")
print("-" * 60)

for word, count in top_words[:10]:
    print(f"{word:15} : {count}")


print("\nNUMPY MATRIX")
print("-" * 60)

print(activity_matrix)


print("\n" + "=" * 60)
print("             END OF GROUPDNA REPORT")
print("=" * 60)

             GROUPDNA
        CAMPUS GROUP ANALYTICS

GROUP OVERVIEW
------------------------------------------------------------
Total Messages : 57
Participants   : 5
First Date     : 2024-05-04
Last Date      : 2024-05-05

MESSAGE COUNT
------------------------------------------------------------
Aarav      : 12
Ananya     : 12
Kabir      : 11
Meera      : 12
Rohan      : 10

ACTIVITY PEAK
------------------------------------------------------------
Busiest Day  : 2024-05-04 ( 34 messages)
Busiest Hour : 14:00 - 15:00

TOP WORDS
------------------------------------------------------------
good            : 7
everyone        : 5
morning         : 4
what            : 3
today           : 3
meeting         : 3
yes             : 3
lets            : 3
night           : 3
project         : 2

NUMPY MATRIX
------------------------------------------------------------
[[0 0 0 0 0 0 0 0 1 2 2 0 1 1 3 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 2 1 2 1 0 3 1 0 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 2 1 